In [2]:
import pandas as pd
import numpy as np
import jdatetime as jdt
import jalali_pandas

In [3]:
mydf = pd.read_excel('WoodInc-sale.xlsx')
mydf.head(n=1)

,تاريخ,شماره,کد مشتري,نام مشتري,کد کالا,نام کالا,واحد,مقدار,في,مبلغ,ماليات,مبلغ خالص
0,13990105,82516,1101328,ابوالفضل تام ...,1-1-04-706,ملامينه 210*280 كد 361 سفيد صابوني كيميا ...,ورق ...,5.0,2339450,11697250,1052753,12750003


تغییر نام ستون ها را انجام میدهم. 

In [4]:
mydf = pd.read_excel('WoodInc-sale.xlsx')
mydf.rename(columns={'تاريخ':'date','نام کالا':'productName','کد کالا':'productID','نام مشتري':'customerName','کد مشتري':'customerID','شماره':'docNumber','مبلغ خالص':'netAmount','ماليات':'tax','مبلغ':'amount','في':'unitPrice','مقدار':'quantity','واحد':'unit'}, inplace=True)
mydf.head()

,date,docNumber,customerID,customerName,productID,productName,unit,quantity,unitPrice,amount,tax,netAmount
0,13990105,82516,1101328,ابوالفضل تام ...,1-1-04-706,ملامينه 210*280 كد 361 سفيد صابوني كيميا ...,ورق ...,5.0,2339450,11697250,1052753,12750003
1,13990105,82516,1101328,ابوالفضل تام ...,1-1-01-505,اچ دي اف 3 ميل 210*244 سفيد پرينتي درجه A ...,ورق ...,2.0,596330,1192660,107339,1299999
2,13990105,82516,1101328,ابوالفضل تام ...,1-1-04-023,ملامينه 204*260 كد 79 آنتيك سويز درجه A ...,ورق ...,5.0,2293578,11467890,1032110,12500000
3,13990105,82517,1100140,احساني عليرضا ...,1-1-05-137,نئوپان 210*366 خام ممتاز گلستان درجه B ...,ورق ...,3.0,1788991,5366973,483028,5850001
4,13990105,82518,1101464,مطهري مهدي ...,1-1-01-294,ام دي اف 183*366 سفيد صابوني كيميا چوب ...,ورق ...,2.0,3669725,7339450,660551,8000001


In [5]:
def to_jalali_date(column, year_dir='L'):
    column = pd.Series(column)

    if column.dtype == 'object':
        sep = next((s for s in ['/', '-', '.'] if s in str(column.iloc[0])), None)
        mod_col = [x.split(sep) for x in column]
    
    elif pd.api.types.is_integer_dtype(column):
        str_col = column.astype(str)
        sample = str_col.iloc[0]
        if len(sample) == 8:
            if year_dir == 'L':
                mod_col = [[x[:4], x[4:6], x[6:]] for x in str_col]
            else:
                mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        elif len(sample) == 6:
            mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        else:
            raise ValueError("Unsupported integer date format")

    else:
        raise TypeError("Unsupported column dtype")

    # Extract year, month, day based on year direction
    idx = (0, 1, 2) if year_dir == 'L' else (2, 1, 0)
    year, month, day = zip(*[(i[idx[0]], i[idx[1]], i[idx[2]]) for i in mod_col])

    # Fix 2-digit years
    year = [
        f"14{y}" if len(y) == 2 and int(y) < 50 else
        f"13{y}" if len(y) == 2 and int(y) >= 50 else
        y
        for y in year
    ]

    return [jdt.date(int(y), int(m), int(d)) for y, m, d in zip(year, month, day)]

In [6]:
mydf['jalaliDate'] = to_jalali_date(mydf['date'])

In [7]:
mydf[['date','jalaliDate']].head()

,date,jalaliDate
0,13990105,1399-01-05
1,13990105,1399-01-05
2,13990105,1399-01-05
3,13990105,1399-01-05
4,13990105,1399-01-05


In [8]:
mydf['month'] = mydf['jalaliDate'].jalali.month
mydf['year'] = mydf['jalaliDate'].jalali.year



In [9]:
monthly_report = mydf.groupby('month').agg(
    total_sale = ('netAmount','sum'),
    average_price = ('unitPrice','mean'),
    customer_count = ('customerID', 'nunique')
).reset_index()


In [10]:
monthly_report

,month,total_sale,average_price,customer_count
0,1,21789957494,1.958121e+06,113
1,2,36217161779,2.006550e+06,161
2,3,30643889582,2.285392e+06,209
3,4,64222149991,2.691524e+06,267
4,5,66578272163,3.196476e+06,237
5,6,59029056413,2.816206e+06,214
6,7,74952726066,3.404953e+06,200
7,8,56156461774,4.061874e+06,158
8,9,73790978679,4.691327e+06,187
9,10,91561359520,4.386980e+06,218


In [19]:
customer_monthly = mydf.groupby(['customerID', 'customerName', 'month']).agg(
    total_buy = ('netAmount', 'sum')
).reset_index()

customer_monthly

,customerID,customerName,month,total_buy
0,1100003,كارگر عليرضا - برقكار ...,11,23600000
1,1100003,كارگر عليرضا - برقكار ...,12,8350000
2,1100004,احسان مولوي ...,9,1586790018
3,1100004,احسان مولوي ...,10,5748889903
4,1100004,احسان مولوي ...,11,7734060200
...,...,...,...,...
2350,1300016,ساختمان در جريان تكميل ...,7,11499998
2351,9900112,محمد كرماني ...,6,19750026
2352,9900113,موسسه شهداي ناجا ...,6,2800000
2353,9900115,مصطفي كشاورزي شاه آبادي ...,9,3800001
